# Vision-Based Landslide Forecasting: Model Training (Stage 3 & 4)

In this notebook, we will use TensorFlow and Keras to build a Convolutional Neural Network (CNN) that can classify whether a satellite image contains a landslide or not. We will use Transfer Learning with MobileNetV2.

## 1. Install Dependencies
Run this cell once to make sure TensorFlow and other required libraries are installed.

In [ ]:
!pip install tensorflow matplotlib scikit-learn

## 2. Load the Dataset
We will load the images from the `dataset` folder and split them into 80% for training and 20% for validation.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

dataset_dir = 'dataset'
batch_size = 32
img_height = 224
img_width = 224

# 80% Training Data
train_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

# 20% Validation (Testing) Data
val_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

class_names = train_ds.class_names
print("Classes found:", class_names)

## 3. Data Augmentation & Model Architecture
We apply random flips and rotations to the images so the model doesn't memorize them. Then we use MobileNetV2 as our base model.

In [ ]:
# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
])

# Build Model (Transfer Learning with MobileNetV2)
# We use the pre-trained weights from 'imagenet' to learn faster.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze base model initially

inputs = tf.keras.Input(shape=(img_height, img_width, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x) # 1 Output node (Landslide or Not)

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

model.summary()

## 4. Train the Model
We train the model for 10 epochs (rounds). You can increase this later if needed.

In [ ]:
epochs = 10
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

## 5. Evaluate and Save
Let's visualize the accuracy over time and save the model.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(epochs)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# Save the final model so we can use it in our UI later
model.save('landslide_model.keras')
print("Model saved successfully as 'landslide_model.keras'")